Install miniconda/anaconda: https://www.anaconda.com/docs/getting-started/installation
Create and activate your conda environment: 

conda create -n your_env python=3.10 numpy jupyter pandas matplotlib scikit0learn -y
conda activate your_env

Make sure to install transformers, and pytorch in your conda environment.

pip install transformers
pip install torch, torchvision

Add "!" before pip if you are running this on Jupyter Notebook.
Installation guide: https://pytorch.org/get-started/locally/
Installation guide: https://huggingface.co/docs/transformers/installation


In [1]:
import torch
import transformers
from math import sqrt

In [2]:
from transformers import AutoTokenizer

AutoTokenizer class has built in models to help us tokenize our text. We will see an example below.

In [32]:
text = 'please check my deep learning model'
model_ckpt = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

We are importing a pre-trained model from the Hugging Face transformers library. One thing you must be careful about is ensuring that your tokenizer matches your pre-trained model; otherwise, the vocabularies won't align. 

Let's tokenize this text above. The tokenizer will convert each word or subword into a number. You can think of this number as an index for that specific word in the model's dictionary. Hence, if you are using a pre-trained Hugging Face model, the tokenizer must match it exactly. Otherwise, your transformer won't understand anything because it's reading a completely different dictionary. It's like me going to a physics exam after reading the chemistry books, lol.

In [33]:
token_id = tokenizer(text, add_special_tokens=False, return_tensors='pt')

In [34]:
print(token_id)

{'input_ids': tensor([[3531, 4638, 2026, 2784, 4083, 2944]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


We asked the tokenizer to tokenize the text and give us the return tensor as a PyTorch tensor object. You can see the exact token numbers by looking at the input_ids inside the dictionary object. Let's take a look.

In [35]:
print(token_id.input_ids)

tensor([[3531, 4638, 2026, 2784, 4083, 2944]])


Before we start building a Transformer model, the first thing we have to do is convert each of these token numbers into a vector of a given dimension; in this case, we will create a vector of 768 dimensions. This is called an embedding vector, which will be used as input in the training of transformers. The reason we are doing this is so we can verify that the sentence and tokens are flowing through each layer of the model without any errors, and we can track the matrix sizes after passing through each layer.

FYI: The Attention Is All You Need paper used an embedding vector of 512, and GPT-3 used 12,288.

In [36]:
import torch.nn as nn
from transformers import AutoConfig
config = AutoConfig.from_pretrained(model_ckpt)

We are directly importing the config file that will give all the bells and whistles of the 'bert-base-uncased model', which you can modify according to your liking. You can import any other model's config file from Hugging Face pre-trained models to train your transformer and modify the model parameters to your liking. 

Just make sure you are not downloading a massive model that was meant to be trained in a hyperscaler data center. We want something small; if you want to experiment with other encoder models, try out 'roberta-base' or 'distilbert-base-uncased'!

In [37]:
config

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.6",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

As you can see, this encoder-only model has a vocabulary size of 30522 and the hidden_size or the embedding vector size of 768.

In [10]:
print(config.hidden_size)

768


In [11]:
print(config.vocab_size)

30522


In [12]:
#Let's convert each of our token to an embedding vector of size 768, and also has a vocabulary of 30522 words.
emb_token = nn.Embedding(num_embeddings=config.vocab_size, embedding_dim=config.hidden_size)
input_embeddings = emb_token(token_id.input_ids)
print(input_embeddings.shape)

torch.Size([1, 6, 768])


We now have a tensor of size (batch size * number of tokens * embedding vector dimension) = (1, 6, 768). 
Here, 1 = one sentence, 6 = 6 tokens (or in this case, our 6 words), and 768 = our embedding vector dimension.

In this document, we will train an encoder-only transformer model. I will keep it simple: we will build it inside out, and then slowly connect each piece. A typical encoder-only model has three major blocks:

Embedding Block -> Self-Attention Block -> Feed-Forward Block

We will build each block separately, and then at the end, connect them with the proper repetition. Let's begin with the Self-Attention block.

Attention Block

First, let's build a typical self-attention block. In simplistic terms, self-attention starts as a matrix multiplication (dot product) between two matrices, Q (queries) and K (keys). We divide this result by the square root of their dimension so the numbers don't blow up and cause problems during training. Next, we apply a softmax function along the last axis to turn those raw scores into our attention weights. Finally, we take those attention weights and do another matrix multiplication with our V (values) matrix. (In the next coding, I will explain exactly how we generate Q, K, and V).

So here is the exact flow:

Matrix multiplication of Q and K transposed -> divide by the square root of the dimension of Q (the dimensions of Q, K, and V are all going to be the same) -> take the softmax on the last axis -> do another matrix multiplication of that output with V.Let's track the matrix sizes to see how this actually works:

Remember, we have 6 words in our sentence. Each of those 6 words will generate its own query, key, and value vector of size 64. When we stack those words together, our Q, K, and V matrices are all sized (6, 64). Why 64? Remember, our embedding dimension was 768, and we will have 12 heads. The Q/K/V matrix sizes for a single head will be (number of tokens, embedding_dimension / num_of_attention_heads), which is exactly (6, 64).

Step 1: We multiply our Q matrix (size 6, 64) by the transpose of our K matrix (size 64, 6).
Step 2: The result is a (6, 6) matrix. We divide this entire matrix by the square root of 64 (which is 8).
Step 3: This (6, 6) output is our attention matrix. Because we have 6 words, we get a 6x6 grid that maps out exactly how much attention every single word should pay to every other word in the sentence. We apply softmax to this (6, 6) matrix so the weights are properly scaled.
Step 4: Finally, we multiply this (6, 6) attention matrix by our V matrix, which has a size of (6, 64). The final math looks like this: (6, 6) * (6, 64) = (6, 64). 

And there it is! Our final output from this single attention block is a (6, 64) matrix. We started with 6 words represented by 64-dimensional vectors, and we end up with 6 updated vectors of the same size—except now, thanks to the self-attention mechanism, every single vector has been infused with the context of the entire sentence!


In [38]:
#let's define a function to build a single head of the self-attention mechanism.
def self_attention(query, key, value):
    dim_head = query.size(-1)
    attn_score = torch.bmm(query, key.traspose(1,2))/sqrt(dim_head)
    attn_weights = nn.functional.softmax(attn_score, dim=-1)
    attn_output = torch.bmm(attn_weights, value)
    return attn_output

In [39]:
class SingleAttentionHead(nn.Module):
    def __int__(self, embedding_dim, head_dim):
        super().__init()
        self.query = nn.Linear(in_features=embedding_dim, out_features=head_dim)
        self.key = nn.Linear(in_features=embedding_dim, out_features=head_dim)
        self.value = nn.Linear(in_features=embedding_dim, out_features=head_dim)

    def forward(self, x):
        head_output = self_attention(query=self.query(x), key=self.key(x), value=self.value(x))
        return head_output

In [40]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        embedding_dim = config.hidden_size  # 768
        num_attn_heads = config.num_attention_heads  # 12
        head_dim = embedding_dim // num_attn_heads   #64
        # '//' is important because it gives an integer; '/' will give a floating decimal

        self.heads = nn.ModuleList(
            [SingleAttentionHead(embedding_dim, head_dim) for _ in range(num_attn_heads)]
        )
        self.multi_head_output = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        x = torch.cat([head(x) for head in self.heads], dim=-1)
        x = self.multi_head_output(x)
        return x 
    

So we built our entire multi-head attention block, let's make sure the output from this block is as expected. It should be same as we entered, which is (1, 6, 768), just to recap: 1 sentence, 6 words/tokens, 768 is embedding dimension.

In [41]:
multi_head_attention = MultiHeadAttention(config)
x = multi_head_attention(input_embeddings)
print(x.shape)

TypeError: SingleAttentionHead.__init__() takes 1 positional argument but 3 were given